> **SUPERSEDED (M1 code rev m1.2-r1).** Use `GEMS_M1_CHAMPION_TO_SUBMISSION.ipynb` — the single canonical Milestone-1 notebook. This notebook is kept for history only; its `EXPECT['code_rev']` (m1.1-r1) no longer matches the code, so it stops at the version check by design.

# M1 · 04 — Leaderboard review (CPU, lightweight)

**Purpose.** After a MANUAL upload, record the public DrivenData score next to the file checksum, recipe and offline validation, and keep an append-only history. Nothing is scraped or submitted automatically, and the public score never changes a selection.

| | |
|---|---|
| Runtime | CPU |
| Prerequisites | notebook 03 produced a `READY_FOR_UPLOAD` file, and you uploaded it by hand |
| Inputs | `submissions/submission_manifest.json`, the uploaded `.tif` (sha256 recomputed), `LEADERBOARD_ENTRY` |
| Outputs | `leaderboard/{leaderboard_history.csv, leaderboard_history.parquet, leaderboard_review.md, leaderboard_history.png (+ _data.csv)}` |
| Next | Milestone 2 (GeoSelect) only after Milestone 1 is complete |

Entries are refused when:
* a field is missing;
* the score is outside [0, 1];
* the date is not ISO;
* the file is not the READY M1 file;
* its sha256 changed;
* the submission id or the file was already recorded.

## 1 · Configuration (fill `LEADERBOARD_ENTRY` after the manual upload)

In [ ]:
CONFIG = {'repo': 'mybenkhadda/GEMS---drivendata', 'ref': 'main', 'drive_root': '/content/drive/MyDrive/GEMS_2',
          'local_root': '/content/gems_work', 'reuse_cached': True, 'strict_provenance': True,
          'overwrite_existing': False, 'device': 'cpu'}
PRIVATE_REPO = False
M1_CONFIG = {}
EXPECT = {'code_rev': 'm1.1-r1', 'commit': None}
LEADERBOARD_ENTRY = {
    'submission_id': None,         # the id shown on the DrivenData submissions page
    'submission_file': None,       # e.g. 'submission_m1_w2_<recipe_id>.tif'
    'public_dti': None,            # the public score, as shown
    'submitted_at': None,          # 'YYYY-MM-DD' (or ISO time)
    'notes': '',
}

## 2 · Drive, clone, version lock, light session

Only Drive and the M1 output folder are needed. The session is attached without the pipeline staging step (`light`).

In [ ]:
import base64, os, subprocess, sys, json
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
CODE = '/content/GEMS'
def git(*a, check=True):
    env = None
    if PRIVATE_REPO:
        from google.colab import userdata
        hdr = base64.b64encode(('x-access-token:' + (userdata.get('GITHUB_TOKEN') or '')).encode()).decode()
        env = {**os.environ, 'GIT_CONFIG_COUNT': '1', 'GIT_CONFIG_KEY_0': 'http.https://github.com/.extraheader', 'GIT_CONFIG_VALUE_0': 'AUTHORIZATION: basic ' + hdr}
    r = subprocess.run(['git', *a], capture_output=True, text=True, env=env)
    if check and r.returncode:
        raise RuntimeError('git ' + ' '.join(a[:4]) + ' failed:\n' + r.stderr[-1500:])
    return r
if not Path(CODE, '.git').exists():
    git('clone', '-q', 'https://github.com/' + CONFIG['repo'] + '.git', CODE)
git('-C', CODE, 'fetch', '-q', '--tags', 'origin')
if git('-C', CODE, 'rev-parse', '--verify', '-q', 'refs/remotes/origin/' + CONFIG['ref'], check=False).returncode == 0:
    git('-C', CODE, 'checkout', '-q', '-B', 'm1_colab_active', 'origin/' + CONFIG['ref'])
else:
    git('-C', CODE, 'checkout', '-q', '--detach', CONFIG['ref'])
HEAD = git('-C', CODE, 'rev-parse', 'HEAD').stdout.strip()
if 'gems_bench' in sys.modules:
    raise RuntimeError('gems_bench already imported: restart the runtime')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', CODE + '[colab]'], check=True)
sys.path.insert(0, CODE)
import gems_bench.leaderboard_focus as LF
assert LF.M1_CODE_REV == EXPECT['code_rev'] and (not EXPECT['commit'] or HEAD.startswith(EXPECT['commit']))
from gems_bench.leaderboard_focus.session import M1Session
m1 = M1Session.light(CONFIG, M1_CONFIG)          # stages submissions/, leaderboard/, manifests/ only
ws = m1.ws
MAN = ws.read_json('submissions/submission_manifest.json') or {}
print('M1 submission:', MAN.get('file'), '| status:', MAN.get('status'), '| sha256:', MAN.get('sha256'))

## 3 · Record (or just review) the history

In [ ]:
from IPython.display import Image, Markdown, display
from gems_bench.leaderboard_focus.reporting import record_leaderboard
LB = record_leaderboard(m1, LEADERBOARD_ENTRY)
for m in LB['messages'] or ['no entry given: history shown only']:
    print(m)
display(LB['history'])
p = ws.path('leaderboard/leaderboard_history.png')
if p.exists():
    display(Image(filename=str(p)))

## 4 · Interpreting the result responsibly

* The public board scores a spatial subset whose chunking is not disclosed. The offline number is a nested pseudo-discovery estimate on 3 zones. A gap of a few hundredths is expected, and |gap| < 0.005 supports no conclusion.
* Never pick the next file because of the public score. The next candidate comes from the offline protocol (Milestone 2).

## 5 · Troubleshooting

* `REJECTED: file sha256 differs`: the uploaded file is not the validated one. Upload the exact file in `submission_sha256.txt`.
* `already recorded`: the history is append-only. Correct a typo by editing the CSV by hand, and note it in `notes`.

## 6 · Next

Milestone 2 (GeoSelect: W2 + fallback fault-segment ranking): see `docs/leaderboard_focus/M1_HANDOFF.md` → entry criteria.